# 第06回 ニューラルネットワーク：値とコードを一段ずつ読む

入力と重みの積を層ごとに読み、1回の更新と時刻ごとの状態変化を区別します。固定ネットワークと学習の小例は別の計算です。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** 1重みの更新はw2→1.75、損失0.5→0.125。系列[1,0]の状態は[1,0.5]。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
import json


In [ ]:
import math


In [ ]:
INPUTS = ((0, 0), (0, 1), (1, 0), (1, 1))


In [ ]:
HIDDEN_WEIGHTS = ((-2, 3), (-2, 1))


In [ ]:
HIDDEN_THRESHOLDS = (-1, 0.5)


In [ ]:
OUTPUT_WEIGHTS = (-60, 94)


In [ ]:
OUTPUT_THRESHOLD = -1


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 積としきい値と層

**今いる段階：** 入力→積→和→活性化→次層

**予想：** しきい値は何回引くか

**操作：** example_inputsを(0, 1)から(1, 1)へ変更し、同じセルを実行します。

**結果の目印：** 各積・合計・中間出力・最終出力が対応する

**戻す：** example_inputs=(0, 1)に戻して再実行します。

**説明する：** 層をつなぐ値は何か


### 計算の定義：`step`

本教材では境界u=0も1とする。


In [ ]:
def step(u):
    """本教材では境界u=0も1とする。"""
    return int(u >= 0)


### 計算の定義：`neuron`

重み付き和を完成させてから、しきい値を一度だけ引く。


In [ ]:
def neuron(inputs, weights, threshold, record=None):
    """重み付き和を完成させてから、しきい値を一度だけ引く。"""
    if len(inputs) != len(weights):
        raise ValueError("input and weight lengths differ")
    products = []
    for index, (value, weight) in enumerate(zip(inputs, weights)):
        product = value * weight
        products.append(product)
        if record is not None:
            record.append((index, value, weight, product))
    u = sum(products) - threshold
    return u, step(u)


### 計算の定義：`forward`

固定の2入力・中間2個・出力1個。途中の値も返す（学習なし）。


In [ ]:
def forward(inputs, hidden_thresholds=HIDDEN_THRESHOLDS):
    """固定の2入力・中間2個・出力1個。途中の値も返す（学習なし）。"""
    if len(inputs) != 2 or len(hidden_thresholds) != 2:
        raise ValueError("this teaching network needs two inputs and thresholds")
    hidden_pairs = [neuron(inputs, weights, threshold)
                    for weights, threshold in zip(HIDDEN_WEIGHTS, hidden_thresholds)]
    hidden = [pair[1] for pair in hidden_pairs]
    output_u, output = neuron(hidden, OUTPUT_WEIGHTS, OUTPUT_THRESHOLD)
    return {"input": list(inputs), "hidden_u": [pair[0] for pair in hidden_pairs],
            "hidden": hidden, "output_u": output_u, "output": output}


### 計算の定義：`lesson_forward`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_forward(inputs=(0,1)):
    tables, rows = [],[]
    for index,(weights,threshold) in enumerate(zip(HIDDEN_WEIGHTS,HIDDEN_THRESHOLDS)):
        products=[]
        u,h=neuron(inputs,weights,threshold,record=products)
        tables.append((f'中間{index}への積',['入力番号','入力値','重み','積'],products))
        rows.append((f'中間{index}',sum(r[3] for r in products),threshold,u,h))
    result=forward(inputs)
    products=[]
    u,o=neuron(result['hidden'],OUTPUT_WEIGHTS,OUTPUT_THRESHOLD,record=products)
    tables.append(('中間から出力への積',['中間番号','中間の出力','重み','積'],products))
    rows.append(('出力',sum(r[3] for r in products),OUTPUT_THRESHOLD,u,o))
    return {'title':'入力×重み→合計−しきい値→出力','question':'しきい値は各入力で毎回引くか',
            'input':list(inputs),'columns':['ニューロン','積の合計','しきい値','引いた後u','u≥0なら1'],'rows':rows,'tables':tables,
            'result':result,'comparison':'入力(0,1)→(1,1)で中間の値と最終出力を比較する。',
            'explanation':'全入力の積を足してから、しきい値を一度だけ引く。この固定ネットワークの観察では重みは学習していない。',
            'reset':'inputs=(0,1)で再実行。'}


In [ ]:
example_inputs = (0, 1)
program_observation_forward = lesson_forward(example_inputs)
learning_show(program_observation_forward, reset='example_inputs=(0, 1)に戻して再実行します。')


## 2. 活性化の比較

**今いる段階：** 同じu→三関数→出力

**予想：** 負の入力はどうなるか

**操作：** example_uを-2から1へ変更し、同じセルを実行します。

**結果の目印：** stepとReLUは0、sigmoidは0より大きい

**戻す：** example_u=-2に戻して再実行します。

**説明する：** 関数の役割の違い


### 計算の定義：`sigmoid`

大きな正負の値でもexpがオーバーフローしない形。


In [ ]:
def sigmoid(u):
    """大きな正負の値でもexpがオーバーフローしない形。"""
    if u >= 0:
        return 1 / (1 + math.exp(-u))
    e = math.exp(u)
    return e / (1 + e)


### 計算の定義：`relu`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def relu(u):
    return max(0, u)


### 計算の定義：`lesson_activation`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_activation(u=-2):
    rows = [(x,step(x),sigmoid(x),relu(x)) for x in [u,0,2]]
    return {'title':'同じuを異なる活性化関数へ渡す','question':'負の値は全ての関数で0になるか',
            'input':f'変更するu={u}。0と2を比較用に併記。',
            'columns':['u','step','sigmoid','ReLU'],'rows':rows,
            'result':{'選択u':u,'step':step(u),'sigmoid':sigmoid(u),'ReLU':relu(u)},
            'comparison':'uを−2→1へ変更。sigmoidは−2でも0より大きい。',
            'explanation':'このstepは境界0を1にする。ReLUは負を0、正をそのまま。sigmoidは滑らかに0〜1へ写す。',
            'reset':'u=-2で再実行。'}


In [ ]:
example_u = -2
program_observation_activation = lesson_activation(example_u)
learning_show(program_observation_activation, reset='example_u=-2に戻して再実行します。')


## 3. 損失から更新

**今いる段階：** 予測→損失→勾配→更新

**予想：** 大きい学習率は必ず良いか

**操作：** example_rateを0.125から1へ変更し、同じセルを実行します。

**結果の目印：** 学習率1では損失4.5へ増える

**戻す：** example_rate=0.125に戻して再実行します。

**説明する：** 勾配と更新方向


### 計算の定義：`gradient_step`

o=w*x, L=(o-target)^2/2 の1回の勾配降下（バイアスなし）。


In [ ]:
def gradient_step(x=2, target=3, weight=2, rate=0.125):
    """o=w*x, L=(o-target)^2/2 の1回の勾配降下（バイアスなし）。"""
    output = weight * x
    gradient = (output - target) * x
    new_weight = weight - rate * gradient
    new_output = new_weight * x
    return {"weight_before": weight, "output_before": output,
            "loss_before": (output - target) ** 2 / 2,
            "gradient": gradient, "rate": rate, "weight_after": new_weight,
            "output_after": new_output, "loss_after": (new_output - target) ** 2 / 2}


### 計算の定義：`lesson_gradient`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_gradient(rate=0.125):
    r=gradient_step(rate=rate)
    rows=[('予測',2,2,'w×x',r['output_before']),('損失',r['output_before'],3,'(出力−正解)^2/2',r['loss_before']),('勾配',r['output_before']-3,2,'誤差×x',r['gradient']),('更新',2,rate,'w−学習率×勾配',r['weight_after']),('更新後予測',r['weight_after'],2,'新w×x',r['output_after']),('更新後損失',r['output_after'],3,'(出力−正解)^2/2',r['loss_after'])]
    return {'title':'1個の重みを1回だけ更新する','question':'学習率を大きくすると必ず良くなるか',
            'input':f'x=2、正解3、初期w=2、学習率={rate}',
            'columns':['段階','使う値1','使う値2','処理','結果'],'rows':rows,
            'result':r,'comparison':'学習率0.125では損失0.5→0.125、1では0.5→4.5。',
            'explanation':'勾配の符号と反対に重みを動かす。大きすぎる更新は目標を行き過ぎる。このセルを再実行しても初期wから1回だけ。',
            'reset':'rate=0.125で再実行。'}


In [ ]:
example_rate = 0.125
program_observation_gradient = lesson_gradient(example_rate)
learning_show(program_observation_gradient, reset='example_rate=0.125に戻して再実行します。')


## 4. 2段の同時更新

**今いる段階：** 順方向→誤差→両勾配→更新

**予想：** どの時点の重みで勾配を求めるか

**操作：** example_chain_rateを0.125から0へ変更し、同じセルを実行します。

**結果の目印：** 勾配a6・b4から同時更新

**戻す：** example_chain_rate=0.125に戻して再実行します。

**説明する：** 順番に古い値と新しい値を混ぜない理由


### 計算の定義：`chain_step`

h=a*x, o=b*h。両勾配を更新前のa,bから求めた後、同時更新する。


In [ ]:
def chain_step(x=1, target=4, a=2, b=3, rate=0.125):
    """h=a*x, o=b*h。両勾配を更新前のa,bから求めた後、同時更新する。"""
    hidden = a * x
    output = b * hidden
    error = output - target
    grad_b = error * hidden
    grad_a = error * b * x
    new_a, new_b = a - rate * grad_a, b - rate * grad_b
    new_output = new_b * (new_a * x)
    return {"hidden_before": hidden, "output_before": output,
            "loss_before": error ** 2 / 2, "grad_a": grad_a, "grad_b": grad_b,
            "a_after": new_a, "b_after": new_b, "output_after": new_output,
            "loss_after": (new_output - target) ** 2 / 2}


### 計算の定義：`lesson_chain`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_chain(rate=0.125):
    r=chain_step(rate=rate)
    return {'title':'2段の計算で勾配を伝える','question':'aを先に変えた値でbの勾配を求めてよいか',
            'input':f'x1、正解4、a2、b3、h=a×x、o=b×h、学習率{rate}',
            'columns':['重み','更新前','勾配の計算','勾配','更新後'],'rows':[('a',2,'(6−4)×3×1',r['grad_a'],r['a_after']),('b',3,'(6−4)×2',r['grad_b'],r['b_after'])],
            'result':r,'comparison':'rate=0ではa・bと損失が変わらない。',
            'explanation':'両勾配は更新前のa・bから求め、その後で同時に更新する。途中hが入力側の重みと出力側をつなぐ。',
            'reset':'rate=0.125で再実行。a2・b3から開始。'}


In [ ]:
example_chain_rate = 0.125
program_observation_chain = lesson_chain(example_chain_rate)
learning_show(program_observation_chain, reset='example_chain_rate=0.125に戻して再実行します。')


## 5. 時間方向の状態

**今いる段階：** 前状態→今の入力→新状態

**予想：** 0入力でも状態が残るか

**操作：** example_recurrent_weightを0.5から0へ変更し、同じセルを実行します。

**結果の目印：** 再帰重み0.5では状態1→0.5

**戻す：** example_recurrent_weight=0.5に戻して再実行します。

**説明する：** 順序と前の状態の役割


### 計算の定義：`recurrent_states`

h_t=ReLU(x_t+r*h_(t-1))。呼び出すたび初期状態から開始する。


In [ ]:
def recurrent_states(sequence, recurrent_weight=0.5, initial_state=0, record=None):
    """h_t=ReLU(x_t+r*h_(t-1))。呼び出すたび初期状態から開始する。"""
    state = initial_state
    states = []
    for time, x in enumerate(sequence):
        previous = state
        carried = recurrent_weight * previous
        total = x + carried
        state = relu(total)
        states.append(state)
        if record is not None:
            record.append((time, x, previous, recurrent_weight, carried, total, state))
    return states


### 計算の定義：`lesson_rnn`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_rnn(recurrent_weight=0.5):
    rows=[]
    states=recurrent_states([1,0],recurrent_weight,record=rows)
    reverse=recurrent_states([0,1],recurrent_weight)
    return {'title':'前の状態を次の時刻へ渡す','question':'同じ0と1でも順序で変わるか',
            'input':f'系列[1,0]、初期状態0、再帰重み={recurrent_weight}',
            'columns':['時刻0始まり','今の入力','前の状態','再帰重み','残る値','合計','ReLU後の状態'],'rows':rows,
            'tables':[('順序比較',['入力列','各時刻の状態'],[('[1,0]',states),('[0,1]',reverse)])],
            'result':states,'comparison':'再帰重み0では、前の状態を次へ引き継がない。',
            'explanation':'同じ関数を時間順に使う。通常の層方向の計算と時間方向の状態更新を区別する。',
            'reset':'recurrent_weight=0.5で再実行。状態は毎回0に戻る。'}


In [ ]:
example_recurrent_weight = 0.5
program_observation_rnn = lesson_rnn(example_recurrent_weight)
learning_show(program_observation_rnn, reset='example_recurrent_weight=0.5に戻して再実行します。')


## 6. 勝者と隣接点の更新

**今いる段階：** 距離→勝者→対象→更新

**予想：** 誰をどれだけ動かすか

**操作：** example_som_rateを0.5から0へ変更し、同じセルを実行します。

**結果の目印：** 距離二乗1・5・9から勝者0

**戻す：** example_som_rate=0.5に戻して再実行します。

**説明する：** 入力空間の距離と地図上の隣接の違い


### 計算の定義：`som_step`

1次元の地図を1回更新。勝者と隣接ノードのみ入力へ近づける。

入力空間は任意次元。地図上の隣接はリストの番号の差が1。
同距離なら先の番号を勝者とする。元の配列を変更しない。


In [ ]:
def som_step(sample, prototypes, rate=0.5, neighbor_factor=0.5):
    """1次元の地図を1回更新。勝者と隣接ノードのみ入力へ近づける。

    入力空間は任意次元。地図上の隣接はリストの番号の差が1。
    同距離なら先の番号を勝者とする。元の配列を変更しない。
    """
    if not sample or not prototypes or any(len(p) != len(sample) for p in prototypes):
        raise ValueError("nonempty vectors of the same dimension are required")
    if not 0 <= rate <= 1 or not 0 <= neighbor_factor <= 1:
        raise ValueError("rate and neighbor_factor must be between 0 and 1")
    distances = [sum((x - w) ** 2 for x, w in zip(sample, p)) for p in prototypes]
    winner = min(range(len(prototypes)), key=lambda i: distances[i])
    updated = []
    for index, prototype in enumerate(prototypes):
        factor = 1 if index == winner else neighbor_factor if abs(index - winner) == 1 else 0
        updated.append([w + rate * factor * (x - w) for x, w in zip(sample, prototype)])
    return {"winner": winner, "squared_distances": distances, "updated": updated}


### 計算の定義：`lesson_som`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_som(rate=0.5):
    sample=[1,0];prototypes=[[0,0],[2,2],[4,0]]
    r=som_step(sample,prototypes,rate=rate)
    rows=[]
    for i,(before,after,distance) in enumerate(zip(prototypes,r['updated'],r['squared_distances'])):
        role='勝者' if i==r['winner'] else '地図上の隣' if abs(i-r['winner'])==1 else '対象外'
        rows.append((i,before,distance,role,after))
    return {'title':'入力に近い代表点とその隣を動かす','question':'全ての代表点を同じだけ動かすか',
            'input':f'入力{sample}、代表点{prototypes}、学習率{rate}',
            'columns':['地図番号','更新前座標','距離の二乗','役割','更新後座標'],'rows':rows,
            'result':r,'comparison':'学習率0では勝者を選ぶが座標は変わらない。',
            'explanation':'距離は入力空間の座標で計算し、隣接は地図番号の差1で決める。勝者は率1、隣は率0.5を学習率へ掛けて入力へ近づける。',
            'reset':'rate=0.5で再実行。代表点を新しく作る。'}


In [ ]:
example_som_rate = 0.5
program_observation_som = lesson_som(example_som_rate)
learning_show(program_observation_som, reset='example_som_rate=0.5に戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`softmax`

最大値を引いてから指数関数を計算する。空列は受け付けない。


In [ ]:
def softmax(scores):
    """最大値を引いてから指数関数を計算する。空列は受け付けない。"""
    if not scores:
        raise ValueError("scores must not be empty")
    maximum = max(scores)
    exponentials = [math.exp(score - maximum) for score in scores]
    total = sum(exponentials)
    return [value / total for value in exponentials]


### 計算の定義：`demo_result`

照合用。超越関数の結果だけ小数6桁へ丸めて表示する。


In [ ]:
def demo_result():
    """照合用。超越関数の結果だけ小数6桁へ丸めて表示する。"""
    return {
        "forward": [forward(x) for x in INPUTS],
        "activations": [{"u": u, "step": step(u), "sigmoid": round(sigmoid(u), 6),
                         "relu": relu(u)} for u in (-2, 0, 2)],
        "softmax_equal": softmax([0, 0]),
        "gradient": gradient_step(),
        "gradient_large_rate": gradient_step(rate=1),
        "chain": chain_step(),
        "rnn_10": recurrent_states([1, 0]),
        "rnn_01": recurrent_states([0, 1]),
        "rnn_no_recurrence": recurrent_states([1, 0], recurrent_weight=0),
        "som": som_step([1, 0], [[0, 0], [2, 2], [4, 0]]),
    }


In [ ]:
import sys
print(sys.executable)
print(sys.version)


## 基準結果

4入力 `(0,0), (0,1), (1,0), (1,1)` に対する出力は `0,1,1,0`。
重みは固定なので、学習した結果ではありません。
1回の勾配更新では損失が `0.5 → 0.125`。RNNは `[1,0] → [1,0.5]`、`[0,1] → [0,1]`。
SOMの更新後は `[[0.5,0], [1.75,1.5], [4,0]]` です。
超越関数のシグモイド値だけ小数6桁に丸めます（表示の丸め誤差は最大0.0000005）。


In [ ]:
baseline_result = demo_result()
for name, value in baseline_result.items():
    print(name, value)


## 実験1：しきい値だけを変更

次のセルは `changed_thresholds` の先頭を `-1` から `-2` に変更した例です。そのまま実行して基準と比較します。
入力 `(1,0)` の中間総入力が `-1 → 0`、最終出力が `1 → 0` になることを確かめます。
元の `HIDDEN_THRESHOLDS` は残します。境界0でステップ関数が1を返すことが理由です。

実行前に変わる値を予想し、実行後は基準との違いと、その理由を説明してください。


In [ ]:
changed_thresholds = (-2, 0.5)
print("基準:", forward([1, 0]))
print("変更:", forward([1, 0], hidden_thresholds=changed_thresholds))


## 実験2：学習率と同時更新

同じ初期値で学習率だけを比べます。学習率1では更新後の損失が4.5へ増えます。
次に2段の計算を確認します。`h=a*x, o=b*h`、正解4、初期値 `x=1, a=2, b=3`。
更新前の両勾配は6と4。学習率0.125で同時更新すると `a=1.25, b=2.5`、損失は0.3828125です。
これは線形な連鎖律の実験で、多層非線形ネットワークの訓練ではありません。

実行前に変わる値を予想し、実行後は基準との違いと、その理由を説明してください。


In [ ]:
for rate in (0, 0.125, 1):
    print(gradient_step(rate=rate))
print("2段:", chain_step())


## 実験3：系列の順序と前の状態

各列は初期状態0で開始します。順序を変えたときの最後の状態を比べます。
再帰係数0では、前の状態からの影響がなくなります。重みの学習は行っていません。

実行前に変わる値を予想し、実行後は基準との違いと、その理由を説明してください。


In [ ]:
for sequence in ([1, 0], [0, 1]):
    print(sequence, "→", recurrent_states(sequence))
print("再帰係数0:", recurrent_states([1, 0], recurrent_weight=0))


## 実験4：SOMの近傍を変更

地図はリストの順にA—B—Cです。勝者は入力空間の距離で選びます。
学習率0.5、近傍係数0.5なので、隣接Bに掛かる係数は0.25。近傍係数を0にするとBは動きません。
比較ごとに同じ元のベクトルを渡します。全体の学習を再現するものではなく、1回の更新です。

実行前に変わる値を予想し、実行後は基準との違いと、その理由を説明してください。


In [ ]:
sample = [1, 0]
prototypes = [[0, 0], [2, 2], [4, 0]]
print("近傍あり:", som_step(sample, prototypes))
print("勝者だけ:", som_step(sample, prototypes, neighbor_factor=0))
print("元のベクトル:", prototypes)


## 結果を言葉にする

- 固定重みでXORが出たことと、学習したことはどう違いますか。
- 勾配が正のとき、なぜ重みから学習率×勾配を引きますか。
- RNNで更新した「状態」と、学習で更新する「重み」はどう違いますか。
- SOMで勝者を選ぶ距離と、近傍を決める距離はどこで測っていますか。

来歴：固定ネットワークは昨年度 `chap05/memo.md` 表5.Aから採用し、コードを数式に合わせて修正。
それ以外の数値例は今年度作成。外部データや個人データはありません。
ボルツマンマシンの学習、RNNの訓練、SOMの反復学習、未知データでの性能評価はこの演習の対象外です。


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python neuralnet.py --explain all`で同じ計算表を表示します。各テーマは`--explain forward`のように指定できます。従来のコマンドと固定期待値も維持しています。
